In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
df.head(3)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


In [3]:
df.shape

(45466, 24)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  str    
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

## Recomendador simple con media ponderada

In [8]:
C = df['vote_average'].mean()
print(C)

5.618207215134184


In [9]:
m = df['vote_count'].quantile(0.90)
print(m)

160.0


In [13]:
q_movies = df.copy().loc[df['vote_count'] >= m]
q_movies.shape

(4555, 24)

In [14]:
# media ponderada
def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']

    # Cálculo de IMDB
    return (v/(v+m) * R) + (m/(m+v) * C)

In [15]:
q_movies['score'] = q_movies.apply(weighted_rating, axis=1)

In [16]:
q_movies = q_movies.sort_values('score', ascending=False)

#Mostrar los primeros 15 resultados
q_movies[['title', 'vote_count', 'vote_average', 'score']].head(15)

,title,vote_count,vote_average,score
314,The Shawshank Redemption,8358.0,8.5,8.445869
834,The Godfather,6024.0,8.5,8.425439
10309,Dilwale Dulhania Le Jayenge,661.0,9.1,8.421453
12481,The Dark Knight,12269.0,8.3,8.265477
2843,Fight Club,9678.0,8.3,8.256385
292,Pulp Fiction,8670.0,8.3,8.251406
522,Schindler's List,4436.0,8.3,8.206639
23673,Whiplash,4376.0,8.3,8.205404
5481,Spirited Away,3968.0,8.3,8.196055
2211,Life Is Beautiful,3643.0,8.3,8.187171


### Interpretación

El recomendador simple ordena las películas usando una media ponderada. Esto permite considerar tanto la calificación promedio como la cantidad de votos recibidos. De esta forma, una película con muy pocos votos no queda por encima de una película con miles de votos y una calificación sólida.

En los resultados aparecen películas reconocidas como *The Shawshank Redemption*, *The Godfather*, *The Dark Knight*, *Fight Club* y *Pulp Fiction*, lo cual indica que el sistema logra generar un ranking razonable de películas populares y bien calificadas.

In [17]:
df['overview'].head()

0    Led by Woody, Andy's toys live happily in his ...
1    When siblings Judy and Peter discover an encha...
2    A family wedding reignites the ancient feud be...
3    Cheated on, mistreated and stepped on, the wom...
4    Just when George Banks has recovered from his ...
Name: overview, dtype: str

## Recomendador basado en contenido con TF-IDF

In [18]:
#Importar TfIdfVectorizer de scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

#Definir el objeto de la clase TF-IDF Vectorizer. Quitamos stop words de inglés
tfidf = TfidfVectorizer(stop_words='english')

#Reemplazar NaN por string vacío
df['overview'] = df['overview'].fillna('')

# Consruir la matriz TF-IDF haciendo ajustes y transformaciones
tfidf_matrix = tfidf.fit_transform(df['overview'])

#Mostrar shape
tfidf_matrix.shape

(45466, 75827)

In [19]:
tfidf.get_feature_names_out()[5000:5010]

array(['avails', 'avaks', 'avalanche', 'avalanches', 'avallone', 'avalon',
       'avant', 'avanthika', 'avanti', 'avaracious'], dtype=object)

In [22]:
from sklearn.metrics.pairwise import linear_kernel

indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def get_recommendations(title):
    idx = indices[title]

    # Calcula similitud solo entre la película elegida y todas las demás
    cosine_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Ordenar de mayor a menor similitud
    sim_scores = list(enumerate(cosine_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Ignorar la primera porque es la misma película
    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]

In [23]:
get_recommendations('The Dark Knight Rises')

12481                                      The Dark Knight
150                                         Batman Forever
1328                                        Batman Returns
15511                           Batman: Under the Red Hood
585                                                 Batman
21194    Batman Unmasked: The Psychology of the Dark Kn...
9230                    Batman Beyond: Return of the Joker
18035                                     Batman: Year One
19792              Batman: The Dark Knight Returns, Part 1
3095                          Batman: Mask of the Phantasm
Name: title, dtype: str

In [24]:
get_recommendations('The Godfather')

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
11297                    Household Saints
34717                   Start Liquidation
10821                            Election
38030            A Mother Should Be Loved
17729                   Short Sharp Shock
26293                  Beck 28 - Familjen
Name: title, dtype: str

### Interpretación del recomendador basado en contenido

El recomendador basado en contenido utiliza la columna `overview`, es decir, la sinopsis de cada película. Primero convierte los textos en vectores numéricos mediante TF-IDF y después calcula la similitud coseno entre la película seleccionada y el resto del catálogo.

En el caso de *The Godfather*, el sistema recomienda principalmente películas relacionadas con la misma saga, como *The Godfather: Part II* y *The Godfather: Part III*. Esto indica que el modelo logra identificar similitudes importantes en la descripción de las películas.

Sin embargo, también aparecen algunas recomendaciones menos conocidas o menos precisas. Esto ocurre porque el modelo solo está usando la sinopsis y no considera todavía otros datos como director, actores, género o popularidad.


## Mapeo inverso

Para poder recomendar películas a partir de un título, necesitamos relacionar cada título con su índice dentro del DataFrame. Esto nos permitirá encontrar la posición de una película específica y compararla con las demás.

En la guía original se calcula una matriz completa de similitud coseno, pero en este caso se ajustó el código para calcular la similitud únicamente cuando se consulta una película. Esto evita problemas de memoria y mantiene la misma lógica del recomendador basado en contenido.

In [25]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

indices[:10]

title
Toy Story                      0
Jumanji                        1
Grumpier Old Men               2
Waiting to Exhale              3
Father of the Bride Part II    4
Heat                           5
Sabrina                        6
Tom and Huck                   7
Sudden Death                   8
GoldenEye                      9
dtype: int64

## Función de recomendación

La función recibe el título de una película, busca su índice y calcula la similitud coseno entre esa película y todas las demás. Después ordena los resultados de mayor a menor similitud, ignora la primera coincidencia porque corresponde a la misma película consultada, y devuelve las 10 películas más similares.

In [26]:
from sklearn.metrics.pairwise import linear_kernel

def get_recommendations(title):
    # Obtener el índice de la película que coincide con el título
    idx = indices[title]

    # Calcular similitud solo entre la película seleccionada y todas las demás
    cosine_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()

    # Convertir los resultados en una lista de tuplas: índice y puntuación
    sim_scores = list(enumerate(cosine_scores))

    # Ordenar las películas según la puntuación de similitud
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Obtener las 10 películas más similares, ignorando la primera porque es la misma película
    sim_scores = sim_scores[1:11]

    # Obtener los índices de las películas recomendadas
    movie_indices = [i[0] for i in sim_scores]

    # Devolver los títulos correspondientes
    return df['title'].iloc[movie_indices]

In [27]:
get_recommendations('The Dark Knight Rises')

12481                                      The Dark Knight
150                                         Batman Forever
1328                                        Batman Returns
15511                           Batman: Under the Red Hood
585                                                 Batman
21194    Batman Unmasked: The Psychology of the Dark Kn...
9230                    Batman Beyond: Return of the Joker
18035                                     Batman: Year One
19792              Batman: The Dark Knight Returns, Part 1
3095                          Batman: Mask of the Phantasm
Name: title, dtype: str

In [28]:
get_recommendations('The Godfather')

1178               The Godfather: Part II
44030    The Godfather Trilogy: 1972-1990
1914              The Godfather: Part III
23126                          Blood Ties
11297                    Household Saints
34717                   Start Liquidation
10821                            Election
38030            A Mother Should Be Loved
17729                   Short Sharp Shock
26293                  Beck 28 - Familjen
Name: title, dtype: str

### Interpretación

El recomendador basado en contenido logró encontrar películas relacionadas a partir de la similitud entre sus sinopsis. En el caso de *The Godfather*, aparecen películas de la misma saga, lo cual indica que el modelo sí identifica relaciones importantes entre los textos.

Sin embargo, el sistema todavía tiene limitaciones, ya que solo utiliza la columna `overview`. Por esa razón, puede recomendar películas similares en descripción, pero no necesariamente películas del mismo director, con los mismos actores o del mismo estilo cinematográfico. Para mejorar esto sería necesario incluir más metadatos, como elenco, director, géneros y palabras clave.

## Mejorando el recomendador

En la guía original se propone mejorar el recomendador usando actores, director, géneros y palabras clave. Sin embargo, en los archivos disponibles para este proyecto no se encontró `credits.csv`, que es el archivo necesario para obtener el elenco y el equipo de producción.

Por esa razón, esta versión mejora el recomendador usando los metadatos disponibles: géneros y palabras clave de la trama. Aunque no incluye actores ni director, permite capturar más información que la sinopsis sola.

In [33]:
# Recargar los datasets desde cero para evitar columnas ya modificadas
df_movies = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
df_keywords = pd.read_csv('./data/keywords.csv')

print(df_movies.shape)
print(df_keywords.shape)

df_movies[['id', 'title', 'genres']].head(3)

(45466, 24)
(46419, 2)


,id,title,genres
0,862,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,8844,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,15602,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."


In [34]:
df_keywords.head(3)

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."


In [35]:
from ast import literal_eval

# Eliminar IDs problemáticos
df_movies = df_movies.drop([19730, 29503, 35587, 35803], errors='ignore').copy()

# Convertir IDs
df_movies['id'] = df_movies['id'].astype('int')
df_keywords['id'] = df_keywords['id'].astype('int')

# Hacer merge
df_meta = df_movies.merge(df_keywords, on='id')

# Convertir columnas tipo texto a listas reales
df_meta['genres'] = df_meta['genres'].apply(literal_eval)
df_meta['keywords'] = df_meta['keywords'].apply(literal_eval)

def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        if len(names) > 3:
            names = names[:3]
        return names
    return []

df_meta['genres'] = df_meta['genres'].apply(get_list)
df_meta['keywords'] = df_meta['keywords'].apply(get_list)

df_meta[['title', 'keywords', 'genres']].head(10)

,title,keywords,genres
0,Toy Story,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,Jumanji,"[board game, disappearance, based on children'...","[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"
3,Waiting to Exhale,"[based on novel, interracial relationship, sin...","[Comedy, Drama, Romance]"
4,Father of the Bride Part II,"[baby, midlife crisis, confidence]",[Comedy]
5,Heat,"[robbery, detective, bank]","[Action, Crime, Drama]"
6,Sabrina,"[paris, brother brother relationship, chauffeur]","[Comedy, Romance]"
7,Tom and Huck,[],"[Action, Adventure, Drama]"
8,Sudden Death,"[terrorist, hostage, explosive]","[Action, Adventure, Thriller]"
9,GoldenEye,"[cuba, falsely accused, secret identity]","[Adventure, Action, Thriller]"


In [36]:
df_keywords.head(5)

,id,keywords
0,862,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,..."
1,8844,"[{'id': 10090, 'name': 'board game'}, {'id': 1..."
2,15602,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392..."
3,31357,"[{'id': 818, 'name': 'based on novel'}, {'id':..."
4,11862,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n..."


In [37]:
from ast import literal_eval

# Recargar ambos archivos desde cero
df_movies = pd.read_csv('./data/movies_metadata.csv', low_memory=False)
df_keywords = pd.read_csv('./data/keywords.csv')

# Eliminar filas problemáticas
df_movies = df_movies.drop([19730, 29503, 35587, 35803], errors='ignore').copy()

# Convertir IDs a número
df_movies['id'] = df_movies['id'].astype('int')
df_keywords['id'] = df_keywords['id'].astype('int')

# Unir películas con keywords
df_meta = df_movies.merge(df_keywords, on='id')

# Convertir columnas de texto a listas reales
df_meta['genres'] = df_meta['genres'].apply(literal_eval)
df_meta['keywords'] = df_meta['keywords'].apply(literal_eval)

# Función para extraer nombres
def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        if len(names) > 3:
            names = names[:3]
        return names
    return []

# Aplicar extracción
df_meta['genres'] = df_meta['genres'].apply(get_list)
df_meta['keywords'] = df_meta['keywords'].apply(get_list)

# Revisar resultado
df_meta[['id', 'title', 'keywords', 'genres']].head(10)

,id,title,keywords,genres
0,862,Toy Story,"[jealousy, toy, boy]","[Animation, Comedy, Family]"
1,8844,Jumanji,"[board game, disappearance, based on children'...","[Adventure, Fantasy, Family]"
2,15602,Grumpier Old Men,"[fishing, best friend, duringcreditsstinger]","[Romance, Comedy]"
3,31357,Waiting to Exhale,"[based on novel, interracial relationship, sin...","[Comedy, Drama, Romance]"
4,11862,Father of the Bride Part II,"[baby, midlife crisis, confidence]",[Comedy]
5,949,Heat,"[robbery, detective, bank]","[Action, Crime, Drama]"
6,11860,Sabrina,"[paris, brother brother relationship, chauffeur]","[Comedy, Romance]"
7,45325,Tom and Huck,[],"[Action, Adventure, Drama]"
8,9091,Sudden Death,"[terrorist, hostage, explosive]","[Action, Adventure, Thriller]"
9,710,GoldenEye,"[cuba, falsely accused, secret identity]","[Adventure, Action, Thriller]"


## Creación de la metadata soup

Para construir el recomendador mejorado, se crea una cadena de texto llamada `soup`, que une las palabras clave y los géneros de cada película. Esta cadena será usada por el vectorizador para calcular similitud entre películas con base en sus metadatos.

In [38]:
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    return []

In [39]:
features = ['keywords', 'genres']

for feature in features:
    df_meta[feature] = df_meta[feature].apply(clean_data)

df_meta[['title', 'keywords', 'genres']].head(10)

,title,keywords,genres
0,Toy Story,"[jealousy, toy, boy]","[animation, comedy, family]"
1,Jumanji,"[boardgame, disappearance, basedonchildren'sbook]","[adventure, fantasy, family]"
2,Grumpier Old Men,"[fishing, bestfriend, duringcreditsstinger]","[romance, comedy]"
3,Waiting to Exhale,"[basedonnovel, interracialrelationship, single...","[comedy, drama, romance]"
4,Father of the Bride Part II,"[baby, midlifecrisis, confidence]",[comedy]
5,Heat,"[robbery, detective, bank]","[action, crime, drama]"
6,Sabrina,"[paris, brotherbrotherrelationship, chauffeur]","[comedy, romance]"
7,Tom and Huck,[],"[action, adventure, drama]"
8,Sudden Death,"[terrorist, hostage, explosive]","[action, adventure, thriller]"
9,GoldenEye,"[cuba, falselyaccused, secretidentity]","[adventure, action, thriller]"


In [40]:
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['genres'])

df_meta['soup'] = df_meta.apply(create_soup, axis=1)

df_meta[['title', 'soup']].head(10)

,title,soup
0,Toy Story,jealousy toy boy animation comedy family
1,Jumanji,boardgame disappearance basedonchildren'sbook ...
2,Grumpier Old Men,fishing bestfriend duringcreditsstinger romanc...
3,Waiting to Exhale,basedonnovel interracialrelationship singlemot...
4,Father of the Bride Part II,baby midlifecrisis confidence comedy
5,Heat,robbery detective bank action crime drama
6,Sabrina,paris brotherbrotherrelationship chauffeur com...
7,Tom and Huck,action adventure drama
8,Sudden Death,terrorist hostage explosive action adventure t...
9,GoldenEye,cuba falselyaccused secretidentity adventure a...


## Vectorizador CountVectorizer

Para el recomendador mejorado se utiliza `CountVectorizer`, ya que en este caso trabajamos con metadatos como géneros y palabras clave. A diferencia de TF-IDF, aquí no buscamos reducir el peso de palabras frecuentes, sino representar la presencia de cada término dentro de la metadata soup.

En la guía original también se incluyen actores y director, pero en este proyecto se trabaja con los metadatos disponibles: `keywords` y `genres`.

In [41]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words='english')

count_matrix = count.fit_transform(df_meta['soup'])

count_matrix.shape

(46480, 9878)

In [42]:
from sklearn.metrics.pairwise import cosine_similarity

indices_meta = pd.Series(df_meta.index, index=df_meta['title']).drop_duplicates()

def get_metadata_recommendations(title):
    idx = indices_meta[title]

    cosine_scores = cosine_similarity(count_matrix[idx], count_matrix).flatten()

    sim_scores = list(enumerate(cosine_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return df_meta['title'].iloc[movie_indices]

In [44]:
get_metadata_recommendations('The Dark Knight Rises')

12501     The Dark Knight
851       Nothing to Lose
4486      An Innocent Man
5014       State Property
6037             Lockdown
6700     Once in the Life
6763       Men of Respect
8078                  PTU
9123        Postman Blues
9243               Shiner
Name: title, dtype: str

In [45]:
get_metadata_recommendations('The Godfather')

28518         In the Name of the Law
164                    Feast of July
260                          L'Enfer
1079                    Bird of Prey
2223                           Belly
4157                   True Believer
6161            Better Luck Tomorrow
7153                      After Life
8376     The Crime of Monsieur Lange
9065                     Golden Gate
Name: title, dtype: str

### Interpretación del recomendador mejorado

Este recomendador utiliza metadatos adicionales, específicamente `keywords` y `genres`, para calcular similitud entre películas. A diferencia del recomendador basado solo en la sinopsis, este modelo considera características más directas de cada película, como sus temas principales y géneros.

En la guía original también se utilizan actores y director, pero en este proyecto no se contó con el archivo `credits.csv`, por lo que la versión implementada se adaptó a los datos disponibles. Aun así, el recomendador mejora la comparación entre películas porque ya no depende únicamente de la descripción textual.